# M04 — Física directa y controles de prismas inducidos

**Level 3 · Prerrequisitos: notebook de fundamentos; lectura: 25 minutos.**

## Objetivos

Ejecute el operador directo de prismas inducidos sobre el fixture sintético congelado. Compare componentes con el oráculo físico Choclo independiente y distinga TMI lineal de anomalía exacta de magnitud. No es inversión de susceptibilidad ni clasificador de remanencia.

## Preparación

Use el entorno pipeline Python 3.12 ya revisado. Este notebook no instala nada.

Importe el código del repositorio y confirme su archivo fuente.

In [ ]:
import runpy,sys
from pathlib import Path
import numpy as np
root=Path.cwd().parents[2];sys.path.insert(0,str(root/'data-pipeline'))
import magnetic_forward
# Reuse the product's frozen synthetic physical control.
fixture=runpy.run_path(str(root/'tests/numerics/test_magnetic_forward.py'))
print(magnetic_forward.ENGINE)

## Datos

Ejecute el operador directo de prismas inducidos sobre el fixture sintético congelado. Compare componentes con el oráculo físico Choclo independiente y distinga TMI lineal de anomalía exacta de magnitud. No es inversión de susceptibilidad ni clasificador de remanencia.

**Files:** tests/numerics/test_magnetic_forward.py, docs/design/features/m04-induced-prism/algorithms.md, docs/design/features/m04-induced-prism/validation.md

## Método

Para I positiva hacia abajo y D horario desde norte, f=(cos I sin D, cos I cos D, −sin I) ENU; B₀=Ff. La magnetización inducida uniforme es M=χ·10⁻⁹B₀/μ₀ A m⁻¹. b secundario da TMI lineal f·b; la anomalía exacta es |B₀+b|−F. Supuestos: campo uniforme, susceptibilidad escalar, sin autodesmagnetización ni remanencia.

Para inclinación I y declinación D, f=(cos I sin D, cos I cos D, −sin I) en este/norte/arriba; Ff es el campo inductor en nT. La susceptibilidad escalar χ es adimensional y la magnetización inducida ideal es M=χ·10⁻⁹Ff/μ₀ A m⁻¹. El campo secundario vectorial integra μ₀/(4π)[3(M·R)R/|R|⁵−M/|R|³] sobre volumen de cada prisma y convierte tesla a nT. TMI lineal es f·b. La anomalía de magnitud exacta es |Ff+b|−F, no |b|. Solo TMI lineal tiene el Jacobiano devuelto respecto de χ. Para |b|≪F, anomalías exacta y lineal se acercan; su diferencia diagnostica el error de aproximación.

Supuestos: campo fuente uniforme, χ escalar por celda, sin remanencia ni autodesmagnetización. El procesador no infiere estos supuestos ni clasifica un levantamiento real. La transecta este agregada es un experimento determinista de modelado basado en el fixture de un prisma del producto. Sus componentes se comparan con la integral física independiente de Choclo; la curva exacta menos lineal cuantifica la aproximación de proyección en este caso. Una diferencia pequeña con el oráculo valida solo ese cálculo controlado, no inversión, validez de fuente de campo ni aceptación del host.


Ejecute el método del producto sobre el control declarado.

In [ ]:
request=fixture['request']()
# Compare actual product response and independent physical prism calculation.
result=magnetic_forward.forward_magnetic(request)
oracle=fixture['physical_oracle'](request)
max_error=float(np.max(np.abs(result['field_components_nt']-oracle)))
print({'componentes_nT':result['field_components_nt'].tolist(),'TMI_lineal_nT':result['linear_tmi_nt'].tolist(),'anomalia_magnitud_nT':result['exact_magnitude_anomaly_nt'].tolist(),'error_componente_maximo_nT':max_error,'aceptado':result['linearization']['full_method_accepted']})

Control adversarial/negativo: aísle un supuesto o incumpla una restricción física declarada.

In [ ]:
from copy import deepcopy
changed=deepcopy(request);changed['inducing_field']['declination_deg']=42.0
# Change only a declared field assumption; this is not field classification.
alternative=magnetic_forward.forward_magnetic(changed)
delta=float(np.max(np.abs(alternative['field_components_nt']-result['field_components_nt'])))
print({'max_component_change_nT':delta})
assert delta>0

## Estado del arte

SimPEG/Geoana es el marco de producción ([Cockett et al., 2015](https://doi.org/10.1016/j.cageo.2015.09.015)); el control del producto lo compara con un cálculo independiente de prisma. Cambiar el campo prueba sensibilidad, no identifica el campo real. Magnitud exacta y TMI lineal difieren; el Jacobiano devuelto corresponde solo a TMI lineal.

Esta transecta sintética determinista usa el operador de producción y un oráculo físico separado. El panel exacta menos lineal es un diagnóstico de aproximación en nT, no un umbral universal ni una anomalía medida.


In [ ]:
from copy import deepcopy
import matplotlib.pyplot as plt

# Recalcular transecta física este con método del producto y oráculo independiente de Choclo.
transecta=deepcopy(request); este=np.linspace(-500.,500.,81)
transecta['receivers_m']=np.column_stack([este,np.zeros_like(este),np.full_like(este,100.)])
p=magnetic_forward.forward_magnetic(transecta); componentes_oraculo=fixture['physical_oracle'](transecta)
err=float(np.max(np.abs(p['field_components_nt']-componentes_oraculo)))
lineal=p['linear_tmi_nt']; exacta=p['exact_magnitude_anomaly_nt']
fig,(ax,bx)=plt.subplots(2,1,figsize=(8.5,7),sharex=True,constrained_layout=True)
ax.plot(este,lineal,label='TMI lineal: f·b',lw=2); ax.plot(este,exacta,'--',label='Exacta: |B₀+b|−F',lw=1.6)
ax.set(ylabel='Anomalía de campo total [nT]',title='Prisma inducido sintético; elevación receptores z=100 m'); ax.legend(); ax.grid(alpha=.25)
bx.plot(este,exacta-lineal,color='#aa3377'); bx.axhline(0,color='.4',lw=.8)
bx.set(xlabel='Este [m]',ylabel='Exacta − lineal [nT]'); bx.grid(alpha=.25)
print({'error_maximo_componentes_vs_oraculo_nT':err,'diferencia_maxima_exacta_lineal_nT':p['linearization']['maximum_abs_difference_nt'],'maxima_razon_secundario_inductor':float(np.max(p['linearization']['secondary_to_background_ratio'])),'metodo_completo_aceptado':p['linearization']['full_method_accepted']})
plt.show()


## Resultados

Use tolerancias independientes congeladas, no valores redondeados. Cambiar declinación debe cambiar componentes. El operador deja falsos origen de campo y aceptación completa; la inversión M04 queda fuera de esta lección.

## Resumen

Mantenga juntas convención, unidades, observables y oráculo. Una imagen directa no es geología recuperada.




## Referencias

- https://doi.org/10.1016/j.cageo.2015.09.015